In [4]:
from google.colab import files
uploaded = files.upload()

Saving aml_dataset_100k_hard.csv to aml_dataset_100k_hard.csv


In [ ]:
# @title
!pip install -U xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 7.3 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 2.0.3
    Uninstalling xgboost-2.0.3:
      Successfully uninstalled xgboost-2.0.3


In [1]:
import xgboost
print(xgboost.__version__)

3.2.0


In [5]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [6]:
df = pd.read_csv("aml_dataset_100k_hard.csv")

df.head()

,in_txn_5m,out_txn_5m,in_amt_5m,out_amt_5m,avg_tx_gap,recv_send_gap,pct_forwarded_60s,in_out_ratio,uniq_senders_10m,uniq_receivers_10m,new_counterparty_pct,region_count,pass_through_ratio,fanin_burst,fanout_burst,sender_region_risk,receiver_region_risk,corridor_multiplier,label
0,9,8,112183,123202.391560,562,230.037389,0.588421,1.111111,6,7,0.954107,2,0.811256,9,9,0.70,1.00,1.0,1
1,11,4,83916,89475.735668,175,291.000000,0.307517,2.400000,9,10,0.723346,4,0.876741,6,9,0.70,0.15,1.0,0
2,9,3,63647,54440.032451,829,442.000000,0.812065,2.500000,10,4,0.734861,4,0.645961,5,6,0.65,0.10,1.0,0
3,4,10,130327,83894.734308,659,133.027087,0.631421,0.454545,5,10,0.878758,4,0.633621,3,10,0.44,0.15,1.0,1
4,6,4,120767,59783.046295,75,132.158594,1.000000,1.400000,6,10,0.158616,2,0.453720,4,6,0.42,0.70,1.0,1


In [7]:
df.tail()

,in_txn_5m,out_txn_5m,in_amt_5m,out_amt_5m,avg_tx_gap,recv_send_gap,pct_forwarded_60s,in_out_ratio,uniq_senders_10m,uniq_receivers_10m,new_counterparty_pct,region_count,pass_through_ratio,fanin_burst,fanout_burst,sender_region_risk,receiver_region_risk,corridor_multiplier,label
99995,3,10,132561,83323.101588,5,97.000000,0.818740,0.363636,2,6,0.877178,3,0.444098,5,9,0.85,0.44,1.0,0
99996,9,10,36817,23302.851101,85,568.000000,0.666201,0.909091,4,11,0.971980,2,0.226039,9,6,0.58,0.80,1.0,0
99997,1,2,82390,55791.515691,138,185.214422,0.557003,0.666667,11,9,0.372518,5,0.171364,1,4,0.05,0.30,1.0,1
99998,0,5,51741,38616.579750,613,550.000000,0.735614,0.166667,4,11,0.857996,4,0.108847,6,8,0.68,0.60,1.0,0
99999,2,7,78925,80277.519765,455,96.143912,0.908395,0.375000,6,5,0.971113,3,0.357101,4,6,0.44,0.80,1.0,1


In [8]:
X = df.drop("label", axis=1)
y = df["label"]

In [ ]:
# import numpy as np

# noise_strength = 0.08

# X_train += np.random.normal(
#     loc=0,
#     scale=noise_strength,
#     size=X_train.shape
# )

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# import numpy as np
# X_train = X_train + np.random.normal(0, 0.02, X_train.shape)

In [10]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=1,
    reg_lambda=4,
    min_child_weight=1,
    gamma=0.5,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.5,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [11]:
y_prob = model.predict_proba(X_test)[:,1]

In [15]:
threshold = 0.30
y_pred = (y_prob > threshold).astype(int)

In [ ]:
y_pred = model.predict(X_test)

In [16]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.77485


In [17]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.73      0.81     13031
           1       0.63      0.87      0.73      6969

    accuracy                           0.77     20000
   macro avg       0.77      0.80      0.77     20000
weighted avg       0.81      0.77      0.78     20000



In [ ]:
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import roc_auc_score

print("AUC:", roc_auc_score(y_test, y_prob))

AUC: 0.9008709806528994


In [18]:
import pandas as pd

normal = pd.DataFrame([[
    2,
    1,
    2000,
    1500,
    300,
    400,
    0.10,
    1.5,
    1,
    1,
    0.20,
    1,
    0.15,
    0,
    0,
    0.15,
    0.15,
    1.0
]], columns=X_train.columns)

proba = model.predict_proba(normal)[0][1]

threshold = 0.3
prediction = 1 if proba >= threshold else 0

print("Fraud probability:", proba)
print("Prediction with threshold 0.3:", prediction)

Fraud probability: 0.0015092965
Prediction with threshold 0.3: 0


In [19]:
import pandas as pd

mule = pd.DataFrame([[
    6,
    5,
    80000,
    76000,
    20,
    15,
    0.85,
    1.1,
    5,
    5,
    0.75,
    3,
    0.9,
    5,
    4,
    0.55,
    0.65,
    1.2
]], columns=X_train.columns)

proba = model.predict_proba(mule)[0][1]

threshold = 0.3
prediction = 1 if proba >= threshold else 0

print("Fraud probability:", proba)
print("Prediction with threshold 0.3:", prediction)

Fraud probability: 0.49446204
Prediction with threshold 0.3: 1


In [20]:
import pandas as pd

fanout = pd.DataFrame([[
    1,
    8,
    50000,
    48000,
    50,
    40,
    0.7,
    0.2,
    1,
    8,
    0.9,
    4,
    0.8,
    1,
    7,
    0.35,
    0.45,
    1.1
]], columns=X_train.columns)

proba = model.predict_proba(fanout)[0][1]

threshold = 0.3
prediction = 1 if proba >= threshold else 0

print("Fraud probability:", proba)
print("Prediction with threshold 0.3:", prediction)

Fraud probability: 0.38865528
Prediction with threshold 0.3: 1


In [22]:
import pandas as pd

fanin = pd.DataFrame([[
    8,
    5,
    90000,
    85000,
    10,
    12,
    0.85,
    6,
    7,
    5,
    0.9,
    3,
    0.85,
    6,
    2,
    0.55,
    0.65,
    1.1
]], columns=X_train.columns)

proba = model.predict_proba(fanin)[0][1]

threshold = 0.3
prediction = 1 if proba >= threshold else 0

print("Fraud probability:", proba)
print("Prediction with threshold 0.3:", prediction)

Fraud probability: 0.5011909
Prediction with threshold 0.3: 1


In [23]:
import joblib

joblib.dump(model, "suspicion-model.pkl")

['suspicion-model.pkl']